# non_linear.ipynb — Teste do módulo `non_linear.py`

Este notebook testa o módulo genérico `non_linear.py`, criado para inversão não linear em problemas geofísicos.

O módulo foi construído para aceitar qualquer função direta no formato:

```python
predicted = forward_function(parameters, *args, **kwargs)
```

Assim, o mesmo módulo pode ser usado para estimar profundidade de prisma, raio/profundidade de esfera, densidade de camada, parâmetros de Moho, parâmetros de flexura, entre outros.

Neste notebook serão testados:

- resíduos e função objetivo;
- Jacobiana por diferenças finitas;
- passo de Gauss-Newton;
- passo de steepest descent;
- passo de Levenberg-Marquardt;
- inversão completa com Levenberg-Marquardt;
- mapa da função objetivo;
- mapa de interação/iteração;
- mapa observado;
- mapa final ajustado;
- mapa de resíduos;
- histograma dos resíduos.

A base conceitual segue a formulação de problemas inversos não lineares, Gauss-Newton, steepest descent e Levenberg-Marquardt discutida por Pujol (2007).


In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# O notebook deve estar no mesmo diretório da pasta "geophysics".
# Dentro dela deve existir o arquivo non_linear.py.
sys.path.append(os.getcwd())

try:
    from geophysics import non_linear
except Exception:
    # Alternativa caso você coloque non_linear.py no mesmo diretório do notebook.
    import non_linear

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["font.size"] = 11

output_dir = "outputs_non_linear"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print("Módulo carregado:", non_linear.__name__)
print("Pasta de saída:", output_dir)


In [ ]:
# Lista das funções disponíveis no módulo
funcoes = [f for f in dir(non_linear) if f.startswith("my_")]

print("Funções disponíveis em non_linear.py:")
for f in funcoes:
    print("-", f)


## Funções auxiliares para plotagem no notebook

O módulo `non_linear.py` não contém funções de plotagem. As figuras serão feitas diretamente no notebook.

In [ ]:
def plot_map(X, Y, data, title, label, filename, cmap="viridis", levels=40, contour=True):
    fig, ax = plt.subplots(figsize=(8, 6))
    c = ax.contourf(X, Y, data, levels=levels, cmap=cmap)
    if contour:
        ct = ax.contour(X, Y, data, levels=12, colors="k", linewidths=0.3)
        ax.clabel(ct, fontsize=7, fmt="%.2f")
    cb = plt.colorbar(c, ax=ax)
    cb.set_label(label)
    ax.set_xlabel("x (km)")
    ax.set_ylabel("y (km)")
    ax.set_title(title)
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    plt.show()


def plot_objective_map(P1, P2, OBJ, xlabel, ylabel, title, filename, path=None, true_params=None):
    fig, ax = plt.subplots(figsize=(8, 6))
    c = ax.contourf(P1, P2, np.log10(OBJ + 1.0e-30), levels=40, cmap="viridis")
    ct = ax.contour(P1, P2, np.log10(OBJ + 1.0e-30), levels=15, colors="k", linewidths=0.3)
    cb = plt.colorbar(c, ax=ax)
    cb.set_label("log10(função objetivo)")

    if path is not None and len(path) > 0:
        ax.plot(path[:, 0], path[:, 1], "w-o", markersize=4, linewidth=1.5, label="iterações")
        ax.plot(path[0, 0], path[0, 1], "rs", markersize=8, label="inicial")
        ax.plot(path[-1, 0], path[-1, 1], "y*", markersize=12, label="final")

    if true_params is not None:
        ax.plot(true_params[0], true_params[1], "r*", markersize=14, label="verdadeiro")

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    plt.show()


def plot_residual_hist(residual, title, filename):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.hist(np.asarray(residual).ravel(), bins=40, edgecolor="black", alpha=0.8)
    ax.set_xlabel("Resíduo")
    ax.set_ylabel("Frequência")
    ax.set_title(title)
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, filename), dpi=300, bbox_inches="tight")
    plt.show()


def print_result(title, result, names, units):
    print("\n" + title)
    print("-"*len(title))
    print("Convergiu:", result["converged"])
    print("Mensagem:", result["message"])
    print("Objetivo final:", result["objective"])
    print("Estatísticas:", result["statistics"])
    report = non_linear.my_parameter_report(result["parameters"], parameter_names=names, units=units)
    for key, item in report.items():
        print(f"{key}: {item['value']:.8g} {item['unit']}")


## Exemplo 1 — Inversão de esfera enterrada

Neste exemplo, estimamos **raio** e **profundidade** de uma esfera enterrada a partir de um mapa sintético de anomalia gravimétrica vertical. O centro horizontal, a densidade e a constante gravitacional são mantidos fixos.

In [ ]:
# Malha de observação em km
nx, ny = 81, 81
x = np.linspace(-20.0, 20.0, nx)
y = np.linspace(-20.0, 20.0, ny)
X, Y = np.meshgrid(x, y)

# Parâmetros verdadeiros
true_radius = 5.0   # km
true_depth = 7.0    # km
density_contrast = 0.25  # g/cm³
xc, yc = 0.0, 0.0

def sphere_gz_forward(parameters, X, Y, density=0.25, xc=0.0, yc=0.0):
    """
    Anomalia gz simplificada de esfera.

    parameters = [radius_km, depth_km]

    Saída em mGal, usando forma compatível com o exemplo didático
    apresentado por Pujol para esfera enterrada.
    """
    radius, depth = parameters
    r2 = (X - xc)**2 + (Y - yc)**2 + depth**2
    gz = (4.0*np.pi/3.0)*6.672*density*(radius**3)*depth/(r2**1.5)
    return gz

# Dados observados sintéticos
rng = np.random.default_rng(42)
gz_true = sphere_gz_forward([true_radius, true_depth], X, Y, density=density_contrast, xc=xc, yc=yc)
gz_obs = gz_true + rng.normal(0.0, 0.03*np.max(gz_true), size=gz_true.shape)

plot_map(X, Y, gz_obs, "Exemplo 1 — Mapa observado: esfera", "gz observado (mGal)", "ex01_observado_esfera.png", cmap="RdYlBu_r")


In [ ]:
# Teste das funções básicas do módulo
initial_sphere = np.array([3.0, 11.0])
pred0 = sphere_gz_forward(initial_sphere, X, Y, density=density_contrast, xc=xc, yc=yc)

res0 = non_linear.my_residual_vector(gz_obs, pred0)
sse0 = non_linear.my_sum_squared_misfit(gz_obs, pred0)
rmse0 = non_linear.my_rmse(gz_obs, pred0)
mae0 = non_linear.my_mae(gz_obs, pred0)
corr0 = non_linear.my_correlation(gz_obs, pred0)
stats0 = non_linear.my_data_statistics(gz_obs, pred0)

print("SSE inicial:", sse0)
print("RMSE inicial:", rmse0)
print("MAE inicial:", mae0)
print("Correlação inicial:", corr0)
print("Estatísticas iniciais:", stats0)


In [ ]:
# Jacobiana e passos linearizados no ponto inicial
args_sphere = (X, Y)
kwargs_sphere = {"density": density_contrast, "xc": xc, "yc": yc}

J0 = non_linear.my_jacobian_finite_difference(
    sphere_gz_forward,
    initial_sphere,
    args=args_sphere,
    kwargs=kwargs_sphere,
    relative_step=1.0e-5
)

r0 = non_linear.my_residual_vector(gz_obs, pred0)

gn_step = non_linear.my_gauss_newton_step(J0, r0)
lm_step = non_linear.my_levenberg_marquardt_step(J0, r0, damping=10.0, scaling=True)
sd_dir = non_linear.my_steepest_descent_direction(J0, r0)
alpha_sd = non_linear.my_optimal_quadratic_step_length(J0, r0, sd_dir)
sd_step = alpha_sd*sd_dir

print("Shape da Jacobiana:", J0.shape)
print("Passo Gauss-Newton:", gn_step)
print("Passo LM escalonado:", lm_step)
print("Passo steepest descent ótimo:", sd_step)


In [ ]:
# Mapa da função objetivo no espaço raio-profundidade
radius_values = np.linspace(2.0, 8.0, 100)
depth_values = np.linspace(3.0, 13.0, 100)

P1, P2, OBJ_sphere = non_linear.my_objective_grid_2d(
    sphere_gz_forward,
    gz_obs,
    radius_values,
    depth_values,
    args=args_sphere,
    kwargs=kwargs_sphere,
    objective="sse"
)

plot_objective_map(
    P1, P2, OBJ_sphere,
    xlabel="Raio (km)",
    ylabel="Profundidade (km)",
    title="Exemplo 1 — Função objetivo: esfera",
    filename="ex01_funcao_objetivo_esfera.png",
    true_params=[true_radius, true_depth]
)


In [ ]:
# Inversão Levenberg-Marquardt
result_sphere = non_linear.my_levenberg_marquardt_inversion(
    sphere_gz_forward,
    gz_obs,
    initial_sphere,
    args=args_sphere,
    kwargs=kwargs_sphere,
    bounds=[(0.5, 15.0), (1.0, 20.0)],
    max_iterations=40,
    damping=100.0,
    scaling=True,
    finite_difference_relative_step=1.0e-5,
    tolerance_parameters=1e-10,
    tolerance_objective=1e-12,
    verbose=True
)

print_result(
    "Resultado da inversão — esfera",
    result_sphere,
    names=["raio", "profundidade"],
    units=["km", "km"]
)


In [ ]:
# Mapa de interação/iteração no espaço de parâmetros
path_sphere = non_linear.my_history_parameters(result_sphere["history"])

plot_objective_map(
    P1, P2, OBJ_sphere,
    xlabel="Raio (km)",
    ylabel="Profundidade (km)",
    title="Exemplo 1 — Mapa de interação/iterações: esfera",
    filename="ex01_interacao_esfera.png",
    path=path_sphere,
    true_params=[true_radius, true_depth]
)


In [ ]:
# Mapas finais: observado, ajustado, resíduo e histograma
gz_fit = result_sphere["predicted"]
gz_res = result_sphere["residual"]

plot_map(X, Y, gz_obs, "Exemplo 1 — Observado", "mGal", "ex01_observado_final.png", cmap="RdYlBu_r")
plot_map(X, Y, gz_fit, "Exemplo 1 — Ajuste final", "mGal", "ex01_ajuste_final.png", cmap="RdYlBu_r")
plot_map(X, Y, gz_res, "Exemplo 1 — Resíduo final", "mGal", "ex01_residuo_final.png", cmap="RdYlBu_r")
plot_residual_hist(gz_res, "Exemplo 1 — Histograma dos resíduos", "ex01_hist_residuos.png")


## Exemplo 2 — Inversão de camada por densidade e espessura efetiva

Neste exemplo, estimamos **densidade equivalente** e **fator de escala de espessura** de uma camada usando a aproximação de placa infinita. Este exemplo mostra que o módulo aceita parâmetros com significado físico diferente.

In [ ]:
# Modelo de espessura conhecida em metros
nx2, ny2 = 101, 101
x2 = np.linspace(-150.0, 150.0, nx2)
y2 = np.linspace(-150.0, 150.0, ny2)
X2, Y2 = np.meshgrid(x2, y2)

thickness_base = 800.0 + 900.0*np.exp(-((X2 - 30.0)**2 + (Y2 + 20.0)**2)/(2*55.0**2))
thickness_base += 200.0*np.sin(2*np.pi*X2/300.0)*np.cos(2*np.pi*Y2/280.0)

true_density = 550.0  # kg/m³
true_scale = 1.25

G_SI = 6.67408e-11
SI2MGAL = 1.0e5

def layer_gravity_forward(parameters, thickness):
    """
    Gravidade de uma camada pela aproximação de placa infinita.

    parameters = [density_kg_m3, thickness_scale]
    """
    density, scale = parameters
    h = scale*thickness
    gz = 2.0*np.pi*G_SI*density*h*SI2MGAL
    return gz

rng = np.random.default_rng(123)
gz_layer_true = layer_gravity_forward([true_density, true_scale], thickness_base)
gz_layer_obs = gz_layer_true + rng.normal(0, 0.02*np.max(gz_layer_true), size=gz_layer_true.shape)

plot_map(X2, Y2, gz_layer_obs, "Exemplo 2 — Mapa observado: camada", "gz observado (mGal)", "ex02_observado_camada.png", cmap="RdYlBu_r")


In [ ]:
# Função objetivo para densidade e escala de espessura
density_values = np.linspace(200.0, 900.0, 120)
scale_values = np.linspace(0.5, 2.0, 120)

P1_layer, P2_layer, OBJ_layer = non_linear.my_objective_grid_2d(
    layer_gravity_forward,
    gz_layer_obs,
    density_values,
    scale_values,
    args=(thickness_base,),
    objective="sse"
)

plot_objective_map(
    P1_layer, P2_layer, OBJ_layer,
    xlabel="Densidade equivalente (kg/m³)",
    ylabel="Fator de espessura",
    title="Exemplo 2 — Função objetivo: camada",
    filename="ex02_funcao_objetivo_camada.png",
    true_params=[true_density, true_scale]
)


In [ ]:
# Inversão da camada
initial_layer = np.array([300.0, 0.8])

result_layer = non_linear.my_levenberg_marquardt_inversion(
    layer_gravity_forward,
    gz_layer_obs,
    initial_layer,
    args=(thickness_base,),
    bounds=[(50.0, 1500.0), (0.1, 4.0)],
    max_iterations=40,
    damping=10.0,
    scaling=True,
    finite_difference_relative_step=1.0e-5,
    tolerance_parameters=1e-10,
    tolerance_objective=1e-12,
    verbose=True
)

print_result(
    "Resultado da inversão — camada",
    result_layer,
    names=["densidade", "fator_espessura"],
    units=["kg/m³", ""]
)


In [ ]:
# Mapa de interação/iteração
path_layer = non_linear.my_history_parameters(result_layer["history"])

plot_objective_map(
    P1_layer, P2_layer, OBJ_layer,
    xlabel="Densidade equivalente (kg/m³)",
    ylabel="Fator de espessura",
    title="Exemplo 2 — Mapa de interação/iterações: camada",
    filename="ex02_interacao_camada.png",
    path=path_layer,
    true_params=[true_density, true_scale]
)


In [ ]:
# Mapas finais
gz_layer_fit = result_layer["predicted"]
gz_layer_res = result_layer["residual"]

plot_map(X2, Y2, gz_layer_obs, "Exemplo 2 — Observado", "mGal", "ex02_observado_final.png", cmap="RdYlBu_r")
plot_map(X2, Y2, gz_layer_fit, "Exemplo 2 — Ajuste final", "mGal", "ex02_ajuste_final.png", cmap="RdYlBu_r")
plot_map(X2, Y2, gz_layer_res, "Exemplo 2 — Resíduo final", "mGal", "ex02_residuo_final.png", cmap="RdYlBu_r")
plot_residual_hist(gz_layer_res, "Exemplo 2 — Histograma dos resíduos", "ex02_hist_residuos.png")


## Exemplo 3 — Inversão de corpo prismático simplificado

Neste exemplo, simulamos um prisma retangular por uma aproximação de massa pontual equivalente. Estimamos **densidade** e **profundidade do centro**. A ideia é mostrar que o módulo funciona também para um problema do tipo prisma/corpo enterrado.

In [ ]:
# Malha em km
nx3, ny3 = 91, 91
x3 = np.linspace(-25.0, 25.0, nx3)
y3 = np.linspace(-25.0, 25.0, ny3)
X3, Y3 = np.meshgrid(x3, y3)

# Dimensões fixas do prisma equivalente em km
Lx, Ly, Lz = 8.0, 10.0, 4.0
volume = Lx*Ly*Lz  # km³

true_prism_density = 0.45  # g/cm³
true_prism_depth = 8.0     # km
x0_prism, y0_prism = 2.0, -3.0

def prism_like_gz_forward(parameters, X, Y, volume, x0=0.0, y0=0.0):
    """
    Aproximação de corpo prismático por massa pontual equivalente.

    parameters = [density_g_cm3, depth_km]
    """
    density, depth = parameters
    r2 = (X - x0)**2 + (Y - y0)**2 + depth**2
    # massa proporcional a density*volume; constante ajustada para mGal em escala didática
    gz = 6.672*density*volume*depth/(r2**1.5)
    return gz

rng = np.random.default_rng(321)
gz_prism_true = prism_like_gz_forward([true_prism_density, true_prism_depth], X3, Y3, volume, x0=x0_prism, y0=y0_prism)
gz_prism_obs = gz_prism_true + rng.normal(0, 0.025*np.max(gz_prism_true), size=gz_prism_true.shape)

plot_map(X3, Y3, gz_prism_obs, "Exemplo 3 — Mapa observado: corpo prismático", "gz observado (mGal)", "ex03_observado_prisma.png", cmap="RdYlBu_r")


In [ ]:
# Função objetivo para densidade e profundidade do prisma
density_prism_values = np.linspace(0.1, 1.0, 120)
depth_prism_values = np.linspace(3.0, 15.0, 120)

P1_prism, P2_prism, OBJ_prism = non_linear.my_objective_grid_2d(
    prism_like_gz_forward,
    gz_prism_obs,
    density_prism_values,
    depth_prism_values,
    args=(X3, Y3, volume),
    kwargs={"x0": x0_prism, "y0": y0_prism},
    objective="sse"
)

plot_objective_map(
    P1_prism, P2_prism, OBJ_prism,
    xlabel="Densidade (g/cm³)",
    ylabel="Profundidade do centro (km)",
    title="Exemplo 3 — Função objetivo: corpo prismático",
    filename="ex03_funcao_objetivo_prisma.png",
    true_params=[true_prism_density, true_prism_depth]
)


In [ ]:
# Inversão do corpo prismático
initial_prism = np.array([0.25, 12.0])

result_prism = non_linear.my_levenberg_marquardt_inversion(
    prism_like_gz_forward,
    gz_prism_obs,
    initial_prism,
    args=(X3, Y3, volume),
    kwargs={"x0": x0_prism, "y0": y0_prism},
    bounds=[(0.01, 2.0), (1.0, 30.0)],
    max_iterations=50,
    damping=10.0,
    scaling=True,
    finite_difference_relative_step=1.0e-5,
    tolerance_parameters=1e-10,
    tolerance_objective=1e-12,
    verbose=True
)

print_result(
    "Resultado da inversão — corpo prismático",
    result_prism,
    names=["densidade", "profundidade"],
    units=["g/cm³", "km"]
)


In [ ]:
# Mapa de interação/iteração
path_prism = non_linear.my_history_parameters(result_prism["history"])

plot_objective_map(
    P1_prism, P2_prism, OBJ_prism,
    xlabel="Densidade (g/cm³)",
    ylabel="Profundidade do centro (km)",
    title="Exemplo 3 — Mapa de interação/iterações: corpo prismático",
    filename="ex03_interacao_prisma.png",
    path=path_prism,
    true_params=[true_prism_density, true_prism_depth]
)


In [ ]:
# Mapas finais
gz_prism_fit = result_prism["predicted"]
gz_prism_res = result_prism["residual"]

plot_map(X3, Y3, gz_prism_obs, "Exemplo 3 — Observado", "mGal", "ex03_observado_final.png", cmap="RdYlBu_r")
plot_map(X3, Y3, gz_prism_fit, "Exemplo 3 — Ajuste final", "mGal", "ex03_ajuste_final.png", cmap="RdYlBu_r")
plot_map(X3, Y3, gz_prism_res, "Exemplo 3 — Resíduo final", "mGal", "ex03_residuo_final.png", cmap="RdYlBu_r")
plot_residual_hist(gz_prism_res, "Exemplo 3 — Histograma dos resíduos", "ex03_hist_residuos.png")


## Comparação dos métodos: Gauss-Newton, Steepest Descent e Levenberg-Marquardt

Agora repetimos o Exemplo 1 com três métodos, para observar as diferenças de convergência.

In [ ]:
methods = [
    ("gauss_newton", "Gauss-Newton"),
    ("steepest_descent", "Steepest Descent"),
    ("levenberg_marquardt", "Levenberg-Marquardt"),
]

results_methods = {}

for method, label in methods:
    print("\nRodando:", label)

    if method == "steepest_descent":
        result = non_linear.my_invert_nonlinear(
            sphere_gz_forward,
            gz_obs,
            initial_sphere,
            args=args_sphere,
            kwargs=kwargs_sphere,
            method=method,
            bounds=[(0.5, 15.0), (1.0, 20.0)],
            max_iterations=60,
            use_line_search=True,
            finite_difference_relative_step=1e-5,
            verbose=False
        )
    else:
        result = non_linear.my_invert_nonlinear(
            sphere_gz_forward,
            gz_obs,
            initial_sphere,
            args=args_sphere,
            kwargs=kwargs_sphere,
            method=method,
            bounds=[(0.5, 15.0), (1.0, 20.0)],
            max_iterations=40,
            damping=100.0,
            scaling=True,
            finite_difference_relative_step=1e-5,
            verbose=False
        )

    results_methods[label] = result
    print(label, result["parameters"], result["objective"], result["message"])


In [ ]:
# Curvas de convergência
fig, ax = plt.subplots(figsize=(8, 6))

for label, result in results_methods.items():
    obj = non_linear.my_extract_history(result["history"], "objective")
    ax.plot(np.arange(len(obj)), obj, "-o", markersize=4, label=label)

ax.set_yscale("log")
ax.set_xlabel("Iteração")
ax.set_ylabel("Função objetivo")
ax.set_title("Comparação de convergência")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "comparacao_convergencia_metodos.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Caminhos no mapa da função objetivo
fig, ax = plt.subplots(figsize=(8, 6))

c = ax.contourf(P1, P2, np.log10(OBJ_sphere + 1e-30), levels=40, cmap="viridis")
plt.colorbar(c, ax=ax, label="log10(função objetivo)")

for label, result in results_methods.items():
    path = non_linear.my_history_parameters(result["history"])
    ax.plot(path[:, 0], path[:, 1], "-o", markersize=4, label=label)

ax.plot(true_radius, true_depth, "r*", markersize=14, label="verdadeiro")
ax.set_xlabel("Raio (km)")
ax.set_ylabel("Profundidade (km)")
ax.set_title("Caminhos de convergência por método")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "comparacao_caminhos_metodos.png"), dpi=300, bbox_inches="tight")
plt.show()


## Salvamento dos resultados principais

In [ ]:
# Salvar históricos principais
for label, result in results_methods.items():
    path = non_linear.my_history_parameters(result["history"])
    obj = non_linear.my_extract_history(result["history"], "objective")
    rmse = non_linear.my_extract_history(result["history"], "rmse")

    n = min(path.shape[0], obj.size, rmse.size)
    table = np.column_stack([np.arange(n), path[:n, 0], path[:n, 1], obj[:n], rmse[:n]])

    fname = "historico_" + label.lower().replace(" ", "_").replace("-", "_") + ".txt"
    np.savetxt(
        os.path.join(output_dir, fname),
        table,
        fmt="%.8e",
        header="iteration parameter1 parameter2 objective rmse"
    )

print("Todos os resultados foram salvos em:", output_dir)
